# Gộp toàn bộ keyframe batch2 thành MỘT file zip (Google Colab)

Hiện keyframe nằm trên Hugging Face thành hơn 600 file zip nhỏ (mỗi video một zip). Notebook này gộp tất cả thành
**một file** `keyframes_batch2_all.zip` để tải bằng một cú bấm. Chạy trên Colab nên không tốn mạng của máy bạn.

**Trước khi chạy:** `Runtime → Change runtime type` (CPU cũng được, GPU không cần). Cần khoảng **95 GB ổ đĩa trống** trên Colab.

| Cell | Việc |
|---|---|
| 1 | Đăng nhập Hugging Face (token có quyền **Write**) |
| 2 | Cấu hình (thường không cần sửa) |
| 3 | Tải các zip nhỏ (song song) → ghi nối vào một file → kiểm tra CRC → upload |

- Bên trong file gộp: `<video>_frame_<số frame>.jpg` (đúng tên như các zip nhỏ, để thư mục sau khi giải nén dùng được ngay) và `manifest.json` (số ảnh của từng video).
- Các zip nhỏ **không bị xóa**, vẫn dùng được như cũ.
- File `keyframes.zip` có sẵn ở gốc repo là keyframe của bộ dữ liệu **cũ**, không liên quan.
- Bị ngắt giữa chừng thì chạy lại cell 3 (làm lại từ đầu, khoảng vài chục phút).


In [ ]:
#@title 1. Đăng nhập Hugging Face
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
from huggingface_hub import login, whoami
login()
print("Đã đăng nhập với tài khoản:", whoami()["name"])


In [ ]:
#@title 2. Cấu hình
REPO = "5clnl/idk"
SOURCE_DIRS = ["keyframes_batch2", "keyframes_S01"]     # nơi có các zip keyframe của từng video
OUT_NAME = "keyframes_batch2_all.zip"                    # tên file gộp trên HF (gốc repo)
DOWNLOAD_WORKERS = 8                                     # số zip tải song song
print(f"Gộp {SOURCE_DIRS} của {REPO} -> {OUT_NAME}")


In [ ]:
#@title 3. Gộp keyframe thành MỘT file zip và upload lên Hugging Face
import itertools, json, os, time, zipfile
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

api = HfApi()
WORK = Path("/content/bundle")
WORK.mkdir(parents=True, exist_ok=True)
OUT_PATH = WORK / OUT_NAME


def log(msg):
    print(msg, flush=True)


# ---- liệt kê các zip keyframe (mỗi video một zip) trên HF
files = api.list_repo_files(REPO, repo_type="dataset")
zips = {}
for f in files:
    for d in SOURCE_DIRS:
        if f.startswith(d + "/") and f.endswith(".zip"):
            v = Path(f).stem
            if v in zips:
                raise RuntimeError(f"video {v} xuất hiện ở cả {zips[v]} và {d}")
            zips[v] = f
order = sorted(zips)
if not order:
    raise RuntimeError("Không thấy zip keyframe nào trên HF")
log(f"{len(order)} zip keyframe (mỗi video một zip) sẽ được gộp vào {OUT_NAME}")

# ---- số ảnh mong đợi của từng video (để phát hiện zip thiếu/hỏng ngay từ đầu)
expected = {}
if "keyframes_batch2/index.json" in files:
    p = hf_hub_download(REPO, "keyframes_batch2/index.json", repo_type="dataset", local_dir=str(WORK / "meta"))
    for v, x in json.load(open(p))["videos"].items():
        expected[v] = x["frames"]
for f in files:
    if f.startswith("keyframes_S01/") and f.endswith(".json"):
        p = hf_hub_download(REPO, f, repo_type="dataset", local_dir=str(WORK / "meta"))
        expected[Path(f).stem] = json.load(open(p))["keyframes"]


def fetch(v):
    for attempt in range(1, 6):
        try:
            return hf_hub_download(REPO, zips[v], repo_type="dataset", local_dir=str(WORK / "dl"))
        except Exception as e:
            log(f"[{v}] tải lỗi lần {attempt}/5: {type(e).__name__}")
            time.sleep(5 * attempt)
    raise RuntimeError(f"không tải được {v}")


# ---- tải song song vài zip một lúc, nhưng GHI vào file gộp theo đúng thứ tự (một luồng ghi)
manifest, n_imgs, n_bytes = {}, 0, 0
tmp_out = OUT_PATH.with_name(OUT_PATH.name + ".tmp")
t0 = time.time()
with ThreadPoolExecutor(DOWNLOAD_WORKERS) as ex, zipfile.ZipFile(tmp_out, "w", zipfile.ZIP_STORED, allowZip64=True) as out:
    it = iter(order)
    pending = deque((v, ex.submit(fetch, v)) for v in itertools.islice(it, DOWNLOAD_WORKERS * 2))
    done_n = 0
    while pending:
        v, fut = pending.popleft()
        p = fut.result()
        nxt = next(it, None)
        if nxt is not None:
            pending.append((nxt, ex.submit(fetch, nxt)))
        with zipfile.ZipFile(p) as zf:
            names = sorted(n for n in zf.namelist() if n.lower().endswith(".jpg"))
            want = expected.get(v)
            if want is not None and len(names) != want:
                raise RuntimeError(f"{v}: zip có {len(names)} ảnh, đáng lẽ phải có {want}")
            if not names:
                raise RuntimeError(f"{v}: zip không có ảnh nào")
            for n in names:
                zi = zf.getinfo(n)
                ni = zipfile.ZipInfo(n, date_time=zi.date_time)
                ni.compress_type = zipfile.ZIP_STORED
                data = zf.read(n)
                out.writestr(ni, data)
                n_bytes += len(data)
            manifest[v] = len(names)
            n_imgs += len(names)
        os.remove(p)
        done_n += 1
        if done_n % 20 == 0 or done_n == len(order):
            log(f"  {done_n}/{len(order)} video | {n_imgs:,} ảnh | {n_bytes / 1e9:.1f} GB | {(time.time() - t0) / 60:.1f} phút")
    out.writestr("manifest.json", json.dumps({"videos": manifest, "total_images": n_imgs}))
os.replace(tmp_out, OUT_PATH)
log(f"Đã gộp: {n_imgs:,} ảnh của {len(manifest)} video -> {OUT_PATH.stat().st_size / 1e9:.1f} GB")

# ---- kiểm tra file gộp TRƯỚC khi upload (đọc lại toàn bộ và kiểm CRC)
log("Đang kiểm tra toàn bộ file gộp (CRC)...")
with zipfile.ZipFile(OUT_PATH) as z:
    bad = z.testzip()
    n_members = len(z.namelist())
if bad is not None:
    raise RuntimeError(f"File gộp bị lỗi ở mục {bad} — chạy lại cell này")
if n_members != n_imgs + 1:
    raise RuntimeError(f"Số mục trong file gộp ({n_members}) khác số ảnh + manifest ({n_imgs + 1})")
log(f"File gộp nguyên vẹn: {n_members - 1:,} ảnh + manifest.json")

# ---- upload (có thử lại)
size = OUT_PATH.stat().st_size
for attempt in range(1, 6):
    try:
        log(f"Upload lần {attempt}: {size / 1e9:.1f} GB ...")
        api.upload_file(path_or_fileobj=str(OUT_PATH), path_in_repo=OUT_NAME, repo_id=REPO, repo_type="dataset",
                        commit_message=f"All batch2 + S01 keyframes in one zip ({n_imgs} images)")
        info = api.get_paths_info(REPO, [OUT_NAME], repo_type="dataset")[0]
        if info.size == size:
            log(f"Upload xong và kích thước khớp ({info.size:,} byte).")
            break
        log(f"Kích thước trên HF ({info.size:,}) khác bản gốc ({size:,}) — thử lại")
    except Exception as e:
        log(f"Upload lỗi: {type(e).__name__}: {str(e)[:200]}")
        time.sleep(30)
else:
    raise RuntimeError("Upload không thành công sau 5 lần")

log("\n================ XONG ================")
log(f"Tải bằng MỘT cú bấm: https://huggingface.co/datasets/{REPO}/resolve/main/{OUT_NAME}")
log(f"Hoặc lệnh (tự nối lại khi đứt mạng):  hf download {REPO} {OUT_NAME} --repo-type dataset --local-dir .")
